# Notebook 01 — Webcam Data Collection

**Project:** Air Guitar CS249R (Software-Only ML Architecture)

This interactive notebook collects labeled hand landmark training data directly from your computer's webcam.

### Objectives:
1. Detect hands and extract 21 3D landmarks via **MediaPipe Hands**.
2. Collect static chord samples for the **Left Hand** (`a`, `am`, `b`, `bm`, `c`, `cm`, `d`, `dm`, `e`, `em`, `f`, `g`, `gm`, `neutral`).
3. Collect temporal motion sequences for the **Right Hand** (`downstroke`, `upstroke`, `idle`).
4. Persist data incrementally to `dataset/raw/chord_landmarks.csv` and `dataset/raw/strumming_frames.csv` with session IDs and timestamps.
5. Provide keyboard controls to pause, record, discard, and switch active labels in real-time.

In [14]:
# 1. Setup and Imports
import os
import sys
import time
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# Add repo root to path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import (
    SUPPORTED_CHORDS, STRUM_CLASSES, RAW_DATA_DIR,
    RAW_CHORD_CSV, RAW_STRUM_CSV, CAMERA_INDEX
)
from src.hand_tracking import HandTracker

print('Configured paths:')
print(f'Raw data directory: {RAW_DATA_DIR}')
print(f'Supported chords ({len(SUPPORTED_CHORDS)}): {SUPPORTED_CHORDS}')
print(f'Strumming classes: {STRUM_CLASSES}')

Configured paths:
Raw data directory: C:\Users\piyus\Desktop\ds cp airguitar\dataset\raw
Supported chords (13): ['a', 'am', 'b', 'bm', 'c', 'cm', 'd', 'dm', 'e', 'em', 'f', 'g', 'gm']
Strumming classes: ['idle', 'downstroke', 'upstroke']


## 2. Interactive Data Collector Class

The collector writes samples incrementally to CSV files so interrupted sessions never lose data.

In [15]:
# 2. Data Collector Implementation
class WebcamDataCollector:
    def __init__(self, camera_idx=CAMERA_INDEX):
        self.camera_idx = camera_idx
        self.tracker = HandTracker()
        self.session_id = f'sess_{int(time.time())}'
        self.raw_dir = RAW_DATA_DIR
        self.raw_dir.mkdir(parents=True, exist_ok=True)
        
        self.chord_csv = RAW_CHORD_CSV
        self.strum_csv = RAW_STRUM_CSV
        self._init_csv_headers()
        
        self.chord_samples_count = 0
        self.strum_samples_count = 0
        
    def _init_csv_headers(self):
        coord_cols = []
        for i in range(21):
            coord_cols.extend([f'x_{i}', f'y_{i}', f'z_{i}'])
            
        if not self.chord_csv.exists() or self.chord_csv.stat().st_size == 0:
            cols = ['timestamp', 'session_id', 'sample_id', 'handedness', 'label', 'confidence'] + coord_cols
            pd.DataFrame(columns=cols).to_csv(self.chord_csv, index=False)
            
        if not self.strum_csv.exists() or self.strum_csv.stat().st_size == 0:
            cols = ['timestamp', 'session_id', 'sequence_id', 'frame_idx', 'handedness', 'label'] + coord_cols
            pd.DataFrame(columns=cols).to_csv(self.strum_csv, index=False)
            
    def record_chord_sample(self, hand_data, label, handedness='Auto'):
        if hand_data is None:
            return False
        lms = hand_data['landmarks'].flatten()
        row = [time.time(), self.session_id, self.chord_samples_count, handedness, label, hand_data['confidence']] + list(lms)
        df = pd.DataFrame([row])
        df.to_csv(self.chord_csv, mode='a', header=False, index=False)
        self.chord_samples_count += 1
        return True
        
    def record_strum_sequence(self, sequence_frames, label, seq_id, handedness='Right'):
        rows = []
        for idx, (ts, lms) in enumerate(sequence_frames):
            row = [ts, self.session_id, seq_id, idx, handedness, label] + list(lms.flatten())
            rows.append(row)
        if rows:
            df = pd.DataFrame(rows)
            df.to_csv(self.strum_csv, mode='a', header=False, index=False)
            self.strum_samples_count += len(rows)
            return True
        return False
        
    def close(self):
        self.tracker.close()

print('WebcamDataCollector ready.')

WebcamDataCollector ready.


## 3. Interactive Collection Routine

Run the cell below to open the webcam window. Keyboard controls:
- **`c`**: Switch chord label.
- **`SPACE`**: Save current hand pose as a chord sample (works with whichever hand is visible).
- **`1` / `2` / `3`**: Select strumming mode (`idle`, `downstroke`, `upstroke`).
- **`r`**: Record 15-frame sequence for selected strum action.
- **`q`** or **`ESC`**: Exit collection.

In [16]:
# 3. Run interactive collection loop
# Set run_collector = True to launch the OpenCV window
run_collector = True

if run_collector:
    collector = WebcamDataCollector()
    cap = cv2.VideoCapture(collector.camera_idx)
    active_chord_idx = 0
    active_strum_label = 'downstroke'
    seq_counter = 0
    feedback_msg = ''
    feedback_timer = 0
    feedback_color = (0, 255, 0)
    all_chords = SUPPORTED_CHORDS + ['neutral']
    
    print('Collecting data... Press q to quit.')
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame, hands = collector.tracker.process_frame(frame, mirror=True)
            collector.tracker.draw_landmarks(frame, hands)
            
            cur_chord = all_chords[active_chord_idx]
            h, w, _ = frame.shape
            cv2.putText(frame, f'Chord Label [c]: {cur_chord.upper()} (Samples: {collector.chord_samples_count})',
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(frame, f'Strum Label [1/2/3]: {active_strum_label.upper()} (Total: {collector.strum_samples_count})',
                        (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            
            # Show which hands are currently detected
            detected_txt = 'Hand: ' + (', '.join(hands.keys()) if hands else 'NONE DETECTED')
            txt_color = (0, 255, 0) if hands else (0, 0, 255)
            cv2.putText(frame, detected_txt, (w - 300, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.65, txt_color, 2)
            
            # Visual confirmation popup
            if feedback_timer > 0:
                cv2.putText(frame, feedback_msg, (w // 2 - 160, h // 2), cv2.FONT_HERSHEY_DUPLEX, 1.0, feedback_color, 2)
                feedback_timer -= 1
                
            cv2.putText(frame, 'SPACE: Save Chord | r: Record Strum Seq | c: Next Chord | q: Quit',
                        (20, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)
                        
            cv2.imshow('Webcam Data Collection', frame)
            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), 27):
                break
            elif key == ord('c'):
                active_chord_idx = (active_chord_idx + 1) % len(all_chords)
            elif key == ord(' '):
                # Accept Left hand if present, otherwise whichever hand is visible
                target_hand = hands.get('Left') or (next(iter(hands.values())) if hands else None)
                handedness_tag = 'Left' if 'Left' in hands else ('Right' if 'Right' in hands else 'Unknown')
                if target_hand is not None:
                    collector.record_chord_sample(target_hand, cur_chord, handedness=handedness_tag)
                    feedback_msg = f'+1 SAVED [{cur_chord.upper()}]'
                    feedback_color = (0, 255, 0)
                    feedback_timer = 15
                    print(f'Recorded chord sample for {cur_chord} (Hand: {handedness_tag}, Total: {collector.chord_samples_count})')
                else:
                    feedback_msg = 'NO HAND VISIBLE!'
                    feedback_color = (0, 0, 255)
                    feedback_timer = 20
                    print('Warning: No hand visible to save sample!')
            elif key == ord('1'):
                active_strum_label = 'idle'
            elif key == ord('2'):
                active_strum_label = 'downstroke'
            elif key == ord('3'):
                active_strum_label = 'upstroke'
            elif key == ord('r'):
                # Capture 15 consecutive frames for strum sequence
                seq_frames = []
                for _ in range(15):
                    r, f = cap.read()
                    if r:
                        _, h = collector.tracker.process_frame(f, mirror=True)
                        strum_h = h.get('Right') or (next(iter(h.values())) if h else None)
                        if strum_h is not None:
                            seq_frames.append((time.time(), strum_h['landmarks']))
                    time.sleep(0.03)
                if len(seq_frames) >= 10:
                    collector.record_strum_sequence(seq_frames, active_strum_label, seq_counter)
                    seq_counter += 1
                    feedback_msg = f'STRUM SEQ #{seq_counter} SAVED'
                    feedback_color = (0, 255, 255)
                    feedback_timer = 18
                    print(f'Recorded strum sequence #{seq_counter} for {active_strum_label}')
    finally:
        cap.release()
        cv2.destroyAllWindows()
        collector.close()
else:
    print('Interactive collection loop skipped. Set run_collector = True to start webcam capture.')

## 4. Dataset Inspection

Examine existing samples in `dataset/raw/` (or synthetic smoke-test dataset if webcam has not been run).

In [17]:
# 4. Check status of collected data
if RAW_CHORD_CSV.exists() and RAW_CHORD_CSV.stat().st_size > 100:
    df_c = pd.read_csv(RAW_CHORD_CSV)
    print(f'Found {len(df_c)} raw chord samples across {df_c["session_id"].nunique()} sessions.')
    print(df_c['label'].value_counts())
else:
    print(f'No raw chord samples yet at {RAW_CHORD_CSV}. Run the collector above or proceed with smoke fixture.')

if RAW_STRUM_CSV.exists() and RAW_STRUM_CSV.stat().st_size > 100:
    df_s = pd.read_csv(RAW_STRUM_CSV)
    print(f'Found {len(df_s)} raw strum frames.')
    print(df_s['label'].value_counts())
else:
    print(f'No raw strumming frames yet at {RAW_STRUM_CSV}.')

Found 871 raw chord samples across 3 sessions.
label
c     91
cm    84
a     72
d     70
bm    69
b     66
gm    65
em    63
g     63
e     61
dm    60
f     56
am    51
Name: count, dtype: int64
Found 2265 raw strum frames.
label
downstroke    840
upstroke      765
idle          660
Name: count, dtype: int64
